In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Iterable

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
from IPython.display import display

In [ ]:
# Beispiel:
SUBJECT_DIRS = [
    Path("/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/home/bstrasser/Projects/Project9_ImplementRecoInICE/Step5_MultiCenterStudy/LargeData_d3hj/Results/Brisbane/Vol03_Dat_NoL2_GradDel/maps"),
    Path("/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/home/bstrasser/Projects/Project9_ImplementRecoInICE/Step5_MultiCenterStudy/LargeData_d3hj/Results/Brisbane/Vol04_Dat_NoL2_GradDel/maps"),
    Path("/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/home/bstrasser/Projects/Project9_ImplementRecoInICE/Step5_MultiCenterStudy/LargeData_d3hj/Results/Brisbane/Vol05_Dat_NoL2_GradDel/maps"),
    Path("/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/home/bstrasser/Projects/Project9_ImplementRecoInICE/Step5_MultiCenterStudy/LargeData_d3hj/Results/Brisbane/Vol07_Dat_NoL2_GradDel/maps"),
    Path("/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/home/bstrasser/Projects/Project9_ImplementRecoInICE/Step5_MultiCenterStudy/LargeData_d3hj/Results/London/Vol01_Dat_NoL2_GradDel/maps"),
    Path("/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/home/bstrasser/Projects/Project9_ImplementRecoInICE/Step5_MultiCenterStudy/LargeData_d3hj/Results/London/Vol02_Dat_NoL2_GradDel/maps"),
    Path("/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/home/bstrasser/Projects/Project9_ImplementRecoInICE/Step5_MultiCenterStudy/LargeData_d3hj/Results/London/Vol03_Dat_NoL2_GradDel/maps"),
    Path("/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/home/bstrasser/Projects/Project9_ImplementRecoInICE/Step5_MultiCenterStudy/LargeData_d3hj/Results/London/Vol04_Dat_NoL2_GradDel/maps"),
    Path("/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/home/bstrasser/Projects/Project9_ImplementRecoInICE/Step5_MultiCenterStudy/LargeData_d3hj/Results/London/Vol05_Dat_NoL2_GradDel/maps"),
    #Path("/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/home/hfischer/ProtonFits/7T/WALINET_7T_MS180_No_Walinet/maps"),
]

# LCModel HZPPPM:
# 3 T Proton ungefähr 123.2 Hz/ppm
# 7 T Proton ungefähr 297.2 Hz/ppm
#
# Am besten den exakten LCModel-Wert einsetzen.
HZ_PER_PPM = 297.222931


# Interpretation der auf das Histogramm gelegten Normalverteilung:
#
# "iqr":
#     sigma = IQR
#     Damit entsprechen Median ± 2*IQR genau mu ± 2*sigma.
#
# "normal_equivalent":
#     sigma = IQR / 1.349
#     Das ist die übliche robuste Schätzung der Standardabweichung
#     für normalverteilte Daten.
NORMAL_SIGMA_MODE = "iqr"


# Für die reine Darstellung kann man extreme Ausreißer abschneiden.
# Die Statistik selbst wird trotzdem aus allen Voxeln berechnet.
PLOT_QUANTILES = (0.001, 0.999)

In [ ]:
PARAMETER_CONFIG = {
    "zero_order_phase": {
        "label": "Zero-order phase",
        "filenames": [
            "0_pha_map.nii.gz",
            "0_pha_map.mnc",
        ],
        "original_unit": "deg",
        "converted_unit": "rad",
        "conversion": lambda x, hz_per_ppm: np.deg2rad(x),
    },

    "first_order_phase": {
        "label": "First-order phase",
        "filenames": [
            "1_pha_map.nii.gz",
            "1_pha_map.mnc",
        ],
        "original_unit": "deg/ppm",
        "converted_unit": "rad/Hz",
        "conversion": lambda x, hz_per_ppm: (
            np.deg2rad(x) / hz_per_ppm
        ),
    },

    "fwhm": {
        "label": "FWHM",
        "filenames": [
            "FWHM_map.nii.gz",
            "FWHM_map.mnc",
        ],
        "original_unit": "ppm",
        "converted_unit": "Hz",
        "conversion": lambda x, hz_per_ppm: (
            x * hz_per_ppm
        ),
    },

    "frequency_shift": {
        "label": "Frequency shift",
        "filenames": [
            "shift_map.nii.gz",
            "shift_map.mnc",
        ],
        "original_unit": "ppm",
        "converted_unit": "Hz",
        "conversion": lambda x, hz_per_ppm: (
            x * hz_per_ppm
        ),
    },

    "snr": {
        "label": "SNR",
        "filenames": [
            "SNR_map.nii.gz",
            "SNR_map.mnc",
        ],
        "original_unit": "a.u.",
        "converted_unit": "a.u.",
        "conversion": lambda x, hz_per_ppm: x,
    },
}

In [ ]:
def find_existing_file(
    directory: Path,
    candidates: Iterable[str],
) -> Path:
    """
    Return the first existing file from a list of possible filenames.
    """
    for filename in candidates:
        path = directory / filename
        if path.is_file():
            return path

    candidate_text = "\n".join(
        f"  - {directory / filename}" for filename in candidates
    )
    raise FileNotFoundError(
        "Keine passende Datei gefunden. Getestet wurden:\n"
        f"{candidate_text}"
    )


def load_minc_array(path: Path) -> np.ndarray:
    """
    Load a MINC file as a floating-point NumPy array.
    """
    image = nib.load(str(path))
    return np.asarray(image.get_fdata(dtype=np.float32))


def convert_parameter_units(
    parameter: str,
    values: np.ndarray,
    hz_per_ppm: float,
) -> np.ndarray:
    """
    Convert LCModel output units into convenient simulator units.

    No sign convention is changed here. This function performs only
    a unit conversion.
    """
    values = np.asarray(values, dtype=np.float64)

    if parameter in {"fwhm", "frequency_shift"}:
        # ppm -> Hz
        return values * hz_per_ppm

    if parameter == "zero_order_phase":
        # degrees -> radians
        return np.deg2rad(values)

    if parameter == "first_order_phase":
        # degrees/ppm -> radians/Hz
        return values * np.pi / (180.0 * hz_per_ppm)

    if parameter == "snr":
        return values.copy()

    raise ValueError(f"Unbekannter Parameter: {parameter}")

In [ ]:
import nibabel as nib
import numpy as np
from pathlib import Path


def load_image_array(
    path: str | Path,
) -> np.ndarray:
    path = Path(path)

    image = nib.load(str(path))

    return np.asarray(
        image.get_fdata(),
        dtype=np.float64,
    )

def pool_lcmodel_maps(
    subject_dirs,
    extra_folder="Extra",
):
    pooled_lists = {
        parameter: []
        for parameter in PARAMETER_CONFIG
    }

    subject_info = []

    for subject_dir in map(Path, subject_dirs):
        extra_dir = subject_dir / extra_folder

        subject_row = {
            "subject": subject_dir.name,
        }

        for parameter, config in PARAMETER_CONFIG.items():
            parameter_path = find_existing_file(
                extra_dir,
                config["filenames"],
            )

            parameter_array = load_image_array(
                parameter_path
            )

            valid = (
                np.isfinite(parameter_array)
                & (parameter_array != 0)
            )

            values = parameter_array[valid].astype(
                np.float64,
                copy=False,
            )

            if values.size == 0:
                raise ValueError(
                    f"Keine gültigen Werte für "
                    f"{subject_dir.name}, {parameter}:\n"
                    f"  Datei: {parameter_path}"
                )

            pooled_lists[parameter].append(values)

            subject_row[f"{parameter}_n_voxels"] = (
                values.size
            )
            subject_row[f"{parameter}_file"] = (
                parameter_path.name
            )

        subject_info.append(subject_row)

    pooled_values = {
        parameter: np.concatenate(value_lists)
        for parameter, value_lists in pooled_lists.items()
    }

    subject_info = pd.DataFrame(subject_info)

    return pooled_values, subject_info

In [ ]:
def sigma_from_iqr(
    iqr: float,
    mode: str = "iqr",
) -> float:
    """
    Derive the Gaussian sigma used for plotting.
    """
    if mode == "iqr":
        return iqr

    if mode == "normal_equivalent":
        return iqr / 1.3489795003921634

    raise ValueError(
        "mode muss 'iqr' oder 'normal_equivalent' sein."
    )


def summarize_array(
    values: np.ndarray,
    *,
    parameter: str,
    representation: str,
    unit: str,
    sigma_mode: str,
) -> dict:
    """
    Calculate pooled descriptive statistics.
    """
    values = np.asarray(values, dtype=np.float64)
    values = values[np.isfinite(values)]

    q1, median, q3 = np.percentile(values, [25, 50, 75])
    iqr = q3 - q1
    two_iqr = 2.0 * iqr
    sigma = sigma_from_iqr(iqr, mode=sigma_mode)

    return {
        "parameter": parameter,
        "label": PARAMETER_CONFIG[parameter]["label"],
        "representation": representation,
        "unit": unit,
        "n_voxels": values.size,
        "median": median,
        "q1": q1,
        "q3": q3,
        "iqr": iqr,
        "2_iqr": two_iqr,
        "median_minus_2iqr": median - two_iqr,
        "median_plus_2iqr": median + two_iqr,
        "derived_normal_mean": median,
        "derived_normal_sigma": sigma,
        "minimum": np.min(values),
        "maximum": np.max(values),
    }


def create_summary_table(
    pooled_values: dict[str, np.ndarray],
    *,
    hz_per_ppm: float,
    sigma_mode: str = "iqr",
) -> tuple[pd.DataFrame, dict[tuple[str, str], np.ndarray]]:
    """
    Create statistics in original and converted units.
    """
    rows: list[dict] = []
    all_representations: dict[
        tuple[str, str],
        np.ndarray,
    ] = {}

    for parameter, original_values in pooled_values.items():
        config = PARAMETER_CONFIG[parameter]

        original_values = np.asarray(
            original_values,
            dtype=np.float64,
        )

        converted_values = convert_parameter_units(
            parameter,
            original_values,
            hz_per_ppm,
        )

        all_representations[(parameter, "original")] = (
            original_values
        )
        all_representations[(parameter, "converted")] = (
            converted_values
        )

        rows.append(
            summarize_array(
                original_values,
                parameter=parameter,
                representation="original",
                unit=config["original_unit"],
                sigma_mode=sigma_mode,
            )
        )

        rows.append(
            summarize_array(
                converted_values,
                parameter=parameter,
                representation="converted",
                unit=config["converted_unit"],
                sigma_mode=sigma_mode,
            )
        )

    summary = pd.DataFrame(rows)

    return summary, all_representations

In [ ]:
pooled_values, subject_info = pool_lcmodel_maps(
    SUBJECT_DIRS
)

summary, all_representations = create_summary_table(
    pooled_values,
    hz_per_ppm=HZ_PER_PPM,
    sigma_mode=NORMAL_SIGMA_MODE,
)

In [ ]:
display(summary)

In [ ]:
shift_summary = summary[
    (summary["parameter"] == "frequency_shift")
    & (summary["representation"] == "converted")
].iloc[0]

frequency_shift_mean_hz = 0.0
frequency_shift_std_hz = float(
    shift_summary["derived_normal_sigma"]
)

print(
    "Frequency-shift distribution: "
    f"mean = {frequency_shift_mean_hz:.2f} Hz, "
    f"std = {frequency_shift_std_hz:.2f} Hz"
)